# 02 — Preprocessing Pipeline
Run & verify the full preprocessing pipeline.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path
from src.preprocessing import build_pipeline
from src.config import PROCESSED_DIR

## Run pipeline

In [2]:
data = build_pipeline()
print('Pipeline complete.')
print(f"X_train : {data['X_train'].shape}  dtype={data['X_train'].dtype}")
print(f"X_val   : {data['X_val'].shape}")
print(f"X_test  : {data['X_test'].shape}")
print(f"Features: {len(data['feature_names'])}")

Pipeline complete.
X_train : (113375, 41)  dtype=float32
X_val   : (12598, 41)
X_test  : (22544, 41)
Features: 41


## Label distributions

In [3]:
from collections import Counter
from src.config import CLASS_NAMES

for split in ('train', 'val', 'test'):
    y = data[f'y_cls_{split}']
    print(f'\n{split.upper()} class distribution:')
    for cls_id, name in enumerate(CLASS_NAMES):
        n = (y == cls_id).sum()
        print(f'  {name:<10} {n:>7,}  ({100*n/len(y):.1f}%)')


TRAIN class distribution:
  normal      60,608  (53.5%)
  DoS         41,334  (36.5%)
  Probe       10,490  (9.3%)
  R2L            896  (0.8%)
  U2R             47  (0.0%)

VAL class distribution:
  normal       6,735  (53.5%)
  DoS          4,593  (36.5%)
  Probe        1,166  (9.3%)
  R2L             99  (0.8%)
  U2R              5  (0.0%)

TEST class distribution:
  normal       9,711  (43.1%)
  DoS          7,460  (33.1%)
  Probe        2,421  (10.7%)
  R2L          2,885  (12.8%)
  U2R             67  (0.3%)


## Scaler statistics

In [4]:
import pandas as pd
scaler = data['scaler']
stats = pd.DataFrame({
    'feature': data['feature_names'],
    'mean': scaler.mean_,
    'std': scaler.scale_,
}).sort_values('std', ascending=False)
stats.head(15)

,feature,mean,std
4,src_bytes,50027.181768,6.187738e+06
5,dst_bytes,21646.550748,4.238730e+06
0,duration,287.927991,2.603063e+03
22,count,84.146373,1.144793e+02
32,dst_host_srv_count,115.564472,1.106998e+02
31,dst_host_count,182.111391,9.919667e+01
23,srv_count,27.697517,7.253936e+01
15,num_root,0.300922,2.537106e+01
12,num_compromised,0.278827,2.495114e+01
2,service,31.231303,1.634381e+01


## Cache preprocessed arrays

In [5]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cache_path = PROCESSED_DIR / 'data.pkl'
with open(cache_path, 'wb') as f:
    pickle.dump(data, f)
print(f'Cached to {cache_path}')

Cached to /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../data/processed/data.pkl
